# DEEZER API PULL

11/5/26 script to pull deezer data from their public API.
- have genre
- can pull top 50 artists in each genre (url format: base + /genre + /genre_id + /artists)
- goal: dataframe of top 50 artists per genre

functions:
- get genre
- get artists from genre
- validation
- make dataframe

ok so the top 50 doesnt work well, what we're gonna do is get the top x tracks across all genres, plays, and get the top artist by plays of each days, top x in most genres, yesterdays, past week, past month (when available). lets also make this just deezer

In [ ]:
import argparse
import requests
import pandas as pd
# import random

import io
import logging
import time
from datetime import datetime, timezone

import boto3
from botocore.exceptions import ClientError

In [15]:
MAX_RETRIES = 3 # still need to implement this
url = 'https://api.deezer.com'
session = requests.Session()

In [16]:
# error class so we know where the error comes from further down the line/when the script is run
class DeezerError(Exception):
    pass

# DEFINE FUNCTIONS

In [17]:
def get_genre_data() -> pd.DataFrame:
    try:
        response = session.get(f'{url}/genre', timeout=15)
        response.raise_for_status()
        body = response.json() # the whole response
        if "error" in body:
            raise DeezerError('error in genres data body')
        df = pd.DataFrame(body["data"])[['id', 'name']]
        return df[df['id'] != 0] # no 'all' genre
    except requests.RequestException as e:
        print("request failed:", e)
        raise
    except DeezerError as e:
        print("deezer error:", e)
        raise

genres = get_genre_data()

In [18]:
def get_top_tracks() -> pd.DataFrame:
    try:
        result = []
        for genre_id, genre in zip(genres['id'], genres['name']):
            response = session.get(
                f'{url}/chart/{genre_id}/tracks',
                params={'limit': 100},
                timeout=15,)
            response.raise_for_status()
            body = response.json() # the whole response
            if "error" in body:
                raise DeezerError('error in artists data body')
            # dataframe transformation
            df = pd.DataFrame(body['data'])
            df['Genre'] = genre # add in genre data
            result.append(df)
            # print(len(result))
            # time.sleep(.5)
        return pd.concat(result, ignore_index=True)
    except requests.RequestException as e:
            print("request failed:", e)
            raise
    except DeezerError as e:
            print("deezer error:", e)
            raise

top_tracks = get_top_tracks()

In [19]:
# expanding out the artist column from a dictionary to separate columns to be added back to the tracks dataframe
artist_info_df = pd.json_normalize(top_tracks['artist'])
artist_info_df = artist_info_df.rename(columns={'id':'artist_id',
                                                'name':'artist_name',
                                                'link':'artist_link',
                                                'picture':'artist_picture',
                                                'tracklist':'top_tracklist'}, errors='ignore')[['artist_id','artist_name','artist_link','artist_picture','top_tracklist']]

In [20]:
album_info_df = pd.json_normalize(top_tracks['album'])
album_info_df = album_info_df.rename(columns={'id':'album_id',
                                              'title':'album_title',
                                              'cover':'album_cover',
                                              'tracklist':'album_tracklist'}, errors='ignore')[['album_id','album_title','album_cover','album_tracklist']]

In [21]:
top_tracks_expanded = pd.concat([top_tracks, artist_info_df, album_info_df], axis=1).drop(columns=['artist', 'album'], errors='ignore')
top_tracks_expanded.sample(1)

,id,title,title_short,title_version,link,duration,rank,explicit_lyrics,explicit_content_lyrics,explicit_content_cover,...,Genre,artist_id,artist_name,artist_link,artist_picture,top_tracklist,album_id,album_title,album_cover,album_tracklist
1316,4201662382,Amnesia,Amnesia,,https://www.deezer.com/track/4201662382,235,410684,False,0,0,...,Salsa,270772662,Dalvin La Melodia,https://www.deezer.com/artist/270772662,https://api.deezer.com/artist/270772662/image,https://api.deezer.com/artist/270772662/top?li...,1046059222,La Bachata Volvió,https://api.deezer.com/album/1046059222/image,https://api.deezer.com/album/1046059222/tracks


In [22]:
# deduplicated artist list
artist_list = top_tracks_expanded['artist_id'].unique().tolist()
len(artist_list)

1671

In [ ]:
# changed, rewrite this to get the artists from top tracks and get their number of fans
def get_artist_data() -> pd.DataFrame:
    try:
        result = []
        for artist_id in artist_list: # just testing with one artist for now
            response = session.get(f'{url}/artist/{artist_id}', timeout=15)
            response.raise_for_status()
            body = response.json() # the whole response
            if "error" in body:
                raise DeezerError('error in artists data body')
            result.append(body)
        return pd.DataFrame(result)
    except requests.RequestException as e:
            print("request failed:", e)
            raise
    except DeezerError as e:
            print("deezer error:", e)
            raise

artists_data = get_artist_data()

# DATA VALIDATION

add checks for nulls, missing genres, artist names, etc

to validate:
- genres
    - not much to do here. check for null data
- top_tracks
    - handle nulls, less than 100 tracks in genre
- artists_df
    - handle null number of fans, missing data in join

In [ ]:
# Raised when fetched data fails a check
class DataQualityError(Exception):
    pass

TRACKS_PER_GENRE = 100
MIN_GENRES = 15
MIN_TRACKS = TRACKS_PER_GENRE * 0.8   # a genre below this many tracks is a problem
MAX_NULL_FRACTION = 0.05


def validate_genres(df: pd.DataFrame) -> list[str]:
    problems = []
    if len(df) < MIN_GENRES:
        problems.append(f"Error: genres has {len(df)} rows, expected at least {MIN_GENRES}")
    if df.isna().any().any():
        problems.append("Error: null values in genres df")
    return problems


def validate_tracks(df: pd.DataFrame) -> list[str]:
    problems = []
    per_genre = df.groupby('Genre').size()
    short = per_genre[per_genre < MIN_TRACKS]
    if len(short):
        problems.append(f"Error: top_tracks: under {MIN_TRACKS:.0f} tracks: {short.to_dict()}")
    nulls = df[['id', 'title', 'artist_id', 'artist_name']].isna().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        problems.append(f"top_tracks: nulls in {nulls.to_dict()}")
    return problems


def validate_artists(df: pd.DataFrame) -> list[str]:
    problems = []
    if df.empty:
        return ["artists: no rows"]
    if df[['id', 'name']].isna().any().any():
        problems.append("artists: null id or name")
    if df['nb_fan'].isna().mean() > MAX_NULL_FRACTION:
        problems.append("artists: too many null nb_fan values")
    if df['id'].duplicated().any():
        problems.append("artists: duplicate ids")
    return problems


def validate_all() -> None:
    problems = (validate_genres(genres)
                + validate_tracks(top_tracks_expanded)
                + validate_artists(artists_data))
    if problems:
        raise DataQualityError("\n".join(problems))
    print("Validation passed")

validate_all()

# UPLOAD TO S3

In [25]:
s3 = boto3.client('s3')

def upload_df(df: pd.DataFrame, bucket: str, key: str) -> bool:
    """Write a DataFrame to S3 as Parquet."""
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False)   # needs pyarrow
    buffer.seek(0)
    try:
        s3.upload_fileobj(buffer, bucket, key)
    except ClientError as e:
        logging.error(e)
        return False
    return True

In [26]:
validate_all()   # raises before anything is written if a check fails

BUCKET = 'deezer-api-practice'
load_date = datetime.now(timezone.utc).strftime('%Y-%m-%d')  # computed once so all files share a date

# one dated folder per dataset: <dataset>/dt=<date>/<dataset>.parquet
datasets = {
    'genres': genres,
    'top_tracks': top_tracks_expanded,
    'artists': artists_data,
}

results = {name: upload_df(df, BUCKET, f'{name}/dt={load_date}/{name}.parquet')
           for name, df in datasets.items()}
print(results)  # all should be True
if not all(results.values()):
    raise RuntimeError(f"upload failed: {results}")

validation passed
{'genres': True, 'top_tracks': True, 'artists': True}


# add all this to main so that it's a python script and package

## Before Lambda automation (checklist)

**Code structure**
- [ ] Turn the notebook into `deezer_pull.py` with a `main()` and a thin `lambda_handler(event, context)`; functions take `session`/`genres` as arguments instead of using globals (`get_top_tracks` reads the global `genres`).
- [ ] Remove `random.sample`: decide the artist set deterministically (all charted artists, or top N by track rank) so runs are comparable day to day.
- [ ] Bucket name, tracks per genre, and artist limit come from environment variables, not hard-coded.
- [ ] Remove unused imports (`argparse`) and the `import random` cell; use `logging` instead of `print`.

**Reliability**
- [ ] Implement `MAX_RETRIES` (currently unused): one `get_json()` helper that retries timeouts, HTTP 429/5xx, and Deezer's quota error (`{"error": {"code": 4}}` returned inside a 200 response).
- [ ] Pace calls (Deezer allows about 50 per 5 seconds).
- [ ] Run time: all ~1,670 artists is ~1,670 sequential calls, probably 10+ minutes (estimate, not measured), close to Lambda's 15-minute limit. Measure it; options are a small thread pool under the rate limit, fewer artists, or splitting the job.
- [ ] `upload_df` returns `False` on S3 errors and only logs; in Lambda make it raise so the invocation fails visibly.
- [ ] Upload only after `validate_all()` passes (done in the upload cell); consider writing a marker file last so partial days are detectable.

**Data**
- [ ] Add a `load_date` / `fetched_at` column to each dataset so rows stay traceable once partitions are queried together in Athena.
- [ ] The same track can chart in several genres (about 130 repeats in a run) and that is expected; dedupe in Athena queries, not at ingest.
- [ ] Keep column types stable day to day (Athena reads Parquet schemas per file).

**AWS**
- [ ] Lambda: Python runtime, timeout up to 900 s, memory ~512 MB+ (pandas), log group with retention.
- [ ] `requests` is not in the Lambda runtime: package it. For pandas + pyarrow use the AWS SDK for pandas managed layer (check the ARN for your Region), or write JSON Lines instead to avoid the heavy dependencies.
- [ ] No credentials in code: the function's execution role needs only `s3:PutObject` on `deezer-api-practice/*` plus logging.
- [ ] Schedule with EventBridge Scheduler (daily, UTC); allow a `load_date` in the event for manual re-runs (same-day re-runs overwrite that partition).
- [ ] Alarm on Lambda errors (CloudWatch alarm -> SNS email) so a failed day does not go unnoticed.